# SIPP (PACW) vs Smart Pension

A historical comparison of an SIPP invested 100% in PACW against a Smart Pension workplace pension.

The model gives both approaches the same underlying global equity return history so that the difference reflects the pension and fund charges rather than different investment allocations.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pacw_ocf = 0.0007
smart_ocf = 0.0050
smart_fixed_monthly = 1.50

start_date = pd.Timestamp("1994-05-31")
end_date = pd.Timestamp("2026-06-30")
start_age = 26
employee_rate = 0.05
employer_rate = 0.03
total_pension_rate = employee_rate + employer_rate


In [ ]:
#load monthly returns

history = pd.read_excel(
    "PACW_History_1994_2026.xlsx",
    sheet_name="Combined_PACW_History",
    parse_dates=["date"]
).set_index("date").sort_index()


In [ ]:
#define contributions by age

salary_by_age_band = [
    (22, 29, 33400),
    (30, 39, 42899),
    (40, 49, 47313),
    (50, 59, 45608),
]

def salary_for_age(age):
    for lo, hi, salary in salary_by_age_band:
        if lo <= age <= hi:
            return salary
    raise ValueError(age)

def monthly_contribution_for_age(age):
    return salary_for_age(age) * total_pension_rate / 12


In [ ]:
#apply charges to returns

def monthly_fee_factor(annual_fee):
    return (1 - annual_fee) ** (1 / 12)

history["SIPP (PACW)"] = (
    (1 + history["underlying_real_return_GBP"])
    * monthly_fee_factor(pacw_ocf)
    - 1
)

history["Smart Pension"] = (
    (1 + history["underlying_real_return_GBP"])
    * monthly_fee_factor(smart_ocf)
    - 1
)


In [ ]:
#calculate values using contributions

def pension_paths(data):
    path_index = [start_date] + list(
        data.index[data.index > start_date]
    )
    result = pd.DataFrame(index=path_index)

    for name in ["SIPP (PACW)", "Smart Pension"]:
        value = 0.0
        values = [value]

        for date, monthly_return in data.loc[
            data.index > start_date, name
        ].items():
            age = start_age + (
                (date.year - start_date.year)
                + (date.month - start_date.month) / 12
            )

            contribution = monthly_contribution_for_age(
                int(np.floor(age))
            )

            value *= 1 + monthly_return

            if name == "Smart Pension":
                value -= smart_fixed_monthly

            value += contribution
            values.append(value)

        result[name] = values

    return result

portfolio_paths = pension_paths(history)
ending = portfolio_paths.iloc[-1]


In [ ]:
#make graphs

fig, ax = plt.subplots(figsize=(11, 6.2))

ax.plot(
    portfolio_paths.index,
    portfolio_paths["SIPP (PACW)"],
    label="SIPP - 100% PACW"
)

ax.plot(
    portfolio_paths.index,
    portfolio_paths["Smart Pension"],
    label="Smart Pension"
)

switch_1 = pd.Timestamp("2006-06-30")
switch_2 = pd.Timestamp("2024-03-31")

ax.axvline(
    switch_1,
    linestyle="--",
    linewidth=1
)

ax.axvline(
    switch_2,
    linestyle="--",
    linewidth=1
)

y_top = portfolio_paths.max().max() * 0.97

ax.annotate(
    "MSCI → Solactive",
    xy=(switch_1, y_top),
    xytext=(8, 0),
    textcoords="offset points",
    va="top",
    ha="left",
    fontsize=9
)

ax.annotate(
    "Solactive → PACW",
    xy=(switch_2, y_top),
    xytext=(-8, 0),
    textcoords="offset points",
    va="top",
    ha="right",
    fontsize=9
)

ax.set_ylim(bottom=0)
ax.set_title("SIPP (PACW) vs Smart Pension")
ax.set_ylabel("Value (inflation matched)")
ax.yaxis.set_major_formatter(
    FuncFormatter(
        lambda x, pos: "£0" if x == 0 else f"£{x/1000:.0f}k"
    )
)
ax.grid(False)
ax.legend(frameon=False)

plt.show()


## Results

| Approach | Ending value |
| --- | ---: |
| SIPP (PACW) | £372,512 |
| Smart Pension | £341,463 |
| **Difference** | **£31,049** |

Values are inflation matched.

**Period:** 31 May 1994 to 30 June 2026  
**Return history:** MSCI ACWI GBP Net Return → Solactive GBP NTR → actual PACW


## Graph

![SIPP (PACW) vs Smart Pension](pension_comparison.png)
